# But First, Coffee: Competitive Revenue Intelligence

## 02 — Data Cleaning and Preparation

This notebook transforms the raw competitive datasets into analysis-ready datasets while preserving the original raw workbook unchanged.

### Cleaning Objectives

The cleaning process will:

1. Create working copies of the raw datasets.
2. Standardize brand and branch information.
3. Validate numeric variables.
4. Standardize product and menu categories.
5. Create promotional-pricing variables.
6. Prepare customer-review variables.
7. Standardize customer-experience themes.
8. Prepare dates and rating-volume variables.
9. Validate individual customer ratings.
10. Identify business-key duplicates.
11. Audit missing values.
12. Perform final validation.
13. Export cleaned datasets to `data/cleaned/`.

### Data Integrity Principle

The original workbook stored in `data/raw/` will not be modified.

All transformations will be performed on copies of the raw datasets and exported separately.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_CLEANED = PROJECT_ROOT / "data" / "cleaned"

RAW_FILE = DATA_RAW / "BFC_competitive_data_collection_v7_individual_ratings.xlsx"

DATA_CLEANED.mkdir(parents=True, exist_ok=True)

print(f"Raw file exists: {RAW_FILE.exists()}")
print(f"Cleaned data folder: {DATA_CLEANED}")

Raw file exists: True
Cleaned data folder: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis/data/cleaned


In [3]:
reviews_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Reviews_Raw"
)

menu_prices_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Menu_Prices_Raw"
)

store_footprint_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Store_Footprint"
)

channels_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Channels_Partnerships"
)

menu_breadth_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Menu_Breadth_Snapshots"
)

official_menu_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Official_Portal_Menu"
)

official_capabilities_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Official_Capabilities"
)

individual_ratings_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Individual_Ratings_Raw"
)

print("Raw datasets loaded successfully.")

Raw datasets loaded successfully.


In [4]:
reviews = reviews_raw.copy()
menu_prices = menu_prices_raw.copy()
store_footprint = store_footprint_raw.copy()
channels = channels_raw.copy()
menu_breadth = menu_breadth_raw.copy()
official_menu = official_menu_raw.copy()
official_capabilities = official_capabilities_raw.copy()
individual_ratings = individual_ratings_raw.copy()

print("Working copies created.")

Working copies created.


In [5]:
copy_validation = pd.DataFrame({
    "dataset": [
        "Reviews",
        "Menu Prices",
        "Store Footprint",
        "Channels",
        "Menu Breadth",
        "Official Menu",
        "Official Capabilities",
        "Individual Ratings"
    ],
    "raw_rows": [
        len(reviews_raw),
        len(menu_prices_raw),
        len(store_footprint_raw),
        len(channels_raw),
        len(menu_breadth_raw),
        len(official_menu_raw),
        len(official_capabilities_raw),
        len(individual_ratings_raw)
    ],
    "working_rows": [
        len(reviews),
        len(menu_prices),
        len(store_footprint),
        len(channels),
        len(menu_breadth),
        len(official_menu),
        len(official_capabilities),
        len(individual_ratings)
    ]
})

copy_validation["match"] = (
    copy_validation["raw_rows"]
    == copy_validation["working_rows"]
)

copy_validation

,dataset,raw_rows,working_rows,match
0,Reviews,282,282,True
1,Menu Prices,150,150,True
2,Store Footprint,3,3,True
3,Channels,16,16,True
4,Menu Breadth,3,3,True
5,Official Menu,36,36,True
6,Official Capabilities,15,15,True
7,Individual Ratings,13,13,True


In [6]:
working_datasets = {
    "reviews": reviews,
    "menu_prices": menu_prices,
    "store_footprint": store_footprint,
    "channels": channels,
    "menu_breadth": menu_breadth,
    "official_menu": official_menu,
    "official_capabilities": official_capabilities,
    "individual_ratings": individual_ratings
}

for name, df in working_datasets.items():
    text_columns = df.select_dtypes(include=["object", "string"]).columns
    
    for column in text_columns:
        df[column] = df[column].apply(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

print("Leading and trailing whitespace removed from text fields.")

Leading and trailing whitespace removed from text fields.


In [7]:
for name, df in working_datasets.items():
    if "brand" in df.columns:
        print(f"\n{name}")
        print(df["brand"].value_counts(dropna=False))


reviews
brand
Starbucks                     108
But First, Coffee             104
The Coffee Bean & Tea Leaf     70
Name: count, dtype: int64

menu_prices
brand
But First, Coffee             50
Starbucks                     50
The Coffee Bean & Tea Leaf    50
Name: count, dtype: int64

store_footprint
brand
But First, Coffee                         1
Starbucks Philippines                     1
The Coffee Bean & Tea Leaf Philippines    1
Name: count, dtype: int64

channels
brand
Starbucks Philippines         6
The Coffee Bean & Tea Leaf    6
But First, Coffee             4
Name: count, dtype: int64

menu_breadth
brand
But First, Coffee             2
The Coffee Bean & Tea Leaf    1
Name: count, dtype: int64

official_menu
brand
The Coffee Bean & Tea Leaf    36
Name: count, dtype: int64

official_capabilities
brand
Starbucks Philippines         6
The Coffee Bean & Tea Leaf    6
But First, Coffee             3
Name: count, dtype: int64

individual_ratings
brand
But First, Coffee    13
Nam

In [8]:
brand_mapping = {
    "But First, Coffee": "But First, Coffee",
    "Starbucks": "Starbucks",
    "Starbucks Philippines": "Starbucks",
    "The Coffee Bean & Tea Leaf": "The Coffee Bean & Tea Leaf",
    "The Coffee Bean & Tea Leaf Philippines": "The Coffee Bean & Tea Leaf"
}

for name, df in working_datasets.items():
    if "brand" in df.columns:
        df["brand"] = df["brand"].replace(brand_mapping)

print("Brand names standardized.")

Brand names standardized.


In [9]:
for name, df in working_datasets.items():
    if "brand" in df.columns:
        print(f"\n{name}")
        print(sorted(df["brand"].dropna().unique()))


reviews
['But First, Coffee', 'Starbucks', 'The Coffee Bean & Tea Leaf']

menu_prices
['But First, Coffee', 'Starbucks', 'The Coffee Bean & Tea Leaf']

store_footprint
['But First, Coffee', 'Starbucks', 'The Coffee Bean & Tea Leaf']

channels
['But First, Coffee', 'Starbucks', 'The Coffee Bean & Tea Leaf']

menu_breadth
['But First, Coffee', 'The Coffee Bean & Tea Leaf']

official_menu
['The Coffee Bean & Tea Leaf']

official_capabilities
['But First, Coffee', 'Starbucks', 'The Coffee Bean & Tea Leaf']

individual_ratings
['But First, Coffee']


In [10]:
VALID_BRANDS = {
    "But First, Coffee",
    "Starbucks",
    "The Coffee Bean & Tea Leaf"
}

brand_validation = []

for name, df in working_datasets.items():
    if "brand" in df.columns:
        observed_brands = set(df["brand"].dropna().unique())
        unexpected = observed_brands - VALID_BRANDS

        brand_validation.append({
            "dataset": name,
            "observed_brands": ", ".join(sorted(observed_brands)),
            "unexpected_brand_count": len(unexpected),
            "valid": len(unexpected) == 0
        })

brand_validation_df = pd.DataFrame(brand_validation)

brand_validation_df

,dataset,observed_brands,unexpected_brand_count,valid
0,reviews,"But First, Coffee, Starbucks, The Coffee Bean & Tea Leaf",0,True
1,menu_prices,"But First, Coffee, Starbucks, The Coffee Bean & Tea Leaf",0,True
2,store_footprint,"But First, Coffee, Starbucks, The Coffee Bean & Tea Leaf",0,True
3,channels,"But First, Coffee, Starbucks, The Coffee Bean & Tea Leaf",0,True
4,menu_breadth,"But First, Coffee, The Coffee Bean & Tea Leaf",0,True
5,official_menu,The Coffee Bean & Tea Leaf,0,True
6,official_capabilities,"But First, Coffee, Starbucks, The Coffee Bean & Tea Leaf",0,True
7,individual_ratings,"But First, Coffee",0,True


In [11]:
branch_datasets = {
    "reviews": reviews,
    "menu_prices": menu_prices,
    "menu_breadth": menu_breadth,
    "individual_ratings": individual_ratings
}

for name, df in branch_datasets.items():
    if "branch" in df.columns:
        print(f"\n{name.upper()}")
        print(f"Unique branches: {df['branch'].nunique()}")
        print("-" * 50)

        for branch in sorted(df["branch"].dropna().unique()):
            print(branch)


REVIEWS
Unique branches: 55
--------------------------------------------------
14 Jupiter Makati
515 Shaw
6750 Building
Avida Cityflex Tower
Ayala North Exchange
Bel-Air
Bellagio Residences
Bloc 11
Boni Avenue Mandaluyong
Cash & Carry
Centrio Mall
Chong Hua Cebu
Diversion Road
Eastwood
Fishermall Malabon
Gen. Ordoñez Avenue
Glorietta 5
Governor Pascual Avenue
Greenhills Town Center
Harbor Point Mall
KCC Zamboanga
Kitchen 26th Street
LGC Boulevard
Limketkai Center
Lucky Chinatown
Magliman San Fernando
Maharlika Highway Santo Tomas
Main Square
Malingap Street
Metroplaza Dagupan
Molino
Noble Place Binondo
Quirino Highway
Robinsons La Union
Robinsons Naga
Robinsons Pangasinan
SM Cabanatuan
SM Center Imus
SM City Manila
SM City Roxas
SM City Sucat
SM Fairview
SM Marilao
SM North EDSA
San Isidro Angono
San Pablo
San Pablo Maharlika
San Vicente Urdaneta
Santa Rosa
Savano Park SJDM
Shell Banilad
Spark Place
UP Town Center
V Mapa
Vermosa

MENU_PRICES
Unique branches: 13
-----------------------

In [12]:
for name, df in branch_datasets.items():
    if {"brand", "branch"}.issubset(df.columns):
        summary = (
            df[["brand", "branch"]]
            .drop_duplicates()
            .sort_values(["brand", "branch"])
        )

        print(f"\n{name.upper()}")
        print(summary.to_string(index=False))


REVIEWS
                     brand                        branch
         But First, Coffee          Avida Cityflex Tower
         But First, Coffee       Boni Avenue Mandaluyong
         But First, Coffee           Gen. Ordoñez Avenue
         But First, Coffee       Governor Pascual Avenue
         But First, Coffee        Greenhills Town Center
         But First, Coffee               Lucky Chinatown
         But First, Coffee         Magliman San Fernando
         But First, Coffee               Malingap Street
         But First, Coffee                        Molino
         But First, Coffee               Quirino Highway
         But First, Coffee            Robinsons La Union
         But First, Coffee             San Isidro Angono
         But First, Coffee                     San Pablo
         But First, Coffee           San Pablo Maharlika
         But First, Coffee          San Vicente Urdaneta
         But First, Coffee                    Santa Rosa
         But First, Co

## Branch Name Validation

Branch names were reviewed across the customer-review, menu-pricing, menu-breadth, and individual-rating datasets.

No clear duplicate branch-name variants requiring standardization were identified. Branch names are therefore retained as collected.

The datasets have different branch coverage because they serve different analytical purposes:

- Customer reviews provide broad multi-branch customer-experience evidence.
- Menu pricing provides a structured competitive pricing sample.
- Menu breadth provides supplementary menu evidence.
- Individual ratings represent a limited pilot sample.

Branches will not be merged across datasets unless both the brand and branch identity clearly correspond.

In [13]:
price_summary = (
    menu_prices
    .groupby("brand")
    .agg(
        observations=("regular_price_php", "count"),
        minimum_price=("regular_price_php", "min"),
        maximum_price=("regular_price_php", "max"),
        mean_price=("regular_price_php", "mean"),
        median_price=("regular_price_php", "median"),
        promo_observations=("promo_price_php", "count")
    )
    .round(2)
    .reset_index()
)

price_summary

,brand,observations,minimum_price,maximum_price,mean_price,median_price,promo_observations
0,"But First, Coffee",50,30,219,134.42,130.0,45
1,Starbucks,50,135,315,201.70,195.0,0
2,The Coffee Bean & Tea Leaf,50,100,295,209.70,220.0,0


In [14]:
invalid_regular_prices = menu_prices[
    menu_prices["regular_price_php"].isna()
    | (menu_prices["regular_price_php"] <= 0)
]

invalid_promo_prices = menu_prices[
    menu_prices["promo_price_php"].notna()
    & (menu_prices["promo_price_php"] <= 0)
]

print(f"Invalid regular prices: {len(invalid_regular_prices)}")
print(f"Invalid promotional prices: {len(invalid_promo_prices)}")

Invalid regular prices: 0
Invalid promotional prices: 0


In [15]:
menu_prices["is_promo"] = (
    menu_prices["promo_price_php"].notna()
)

menu_prices["is_promo"].value_counts()

is_promo
False    105
True      45
Name: count, dtype: int64

In [16]:
menu_prices["effective_price_php"] = np.where(
    menu_prices["is_promo"],
    menu_prices["promo_price_php"],
    menu_prices["regular_price_php"]
)

menu_prices[
    [
        "brand",
        "product",
        "regular_price_php",
        "promo_price_php",
        "is_promo",
        "effective_price_php"
    ]
].head(10)

,brand,product,regular_price_php,promo_price_php,is_promo,effective_price_php
0,"But First, Coffee",Spanish Latte,150,120.0,True,120.0
1,"But First, Coffee",Caramel Macchiato,150,120.0,True,120.0
2,"But First, Coffee",Matcha Latte,130,104.0,True,104.0
3,"But First, Coffee",Vietnamese,75,60.0,True,60.0
4,"But First, Coffee",Vietnamese,75,60.0,True,60.0
5,"But First, Coffee",Spanish Latte,130,104.0,True,104.0
6,"But First, Coffee",Caramel Macchiato,130,104.0,True,104.0
7,"But First, Coffee",Matcha Latte,130,104.0,True,104.0
8,"But First, Coffee",Choco Cookie Latte,180,144.0,True,144.0
9,"But First, Coffee",Spanish Latte,130,NaN,False,130.0


In [17]:
menu_prices["discount_php"] = np.where(
    menu_prices["is_promo"],
    menu_prices["regular_price_php"] - menu_prices["promo_price_php"],
    np.nan
)

menu_prices["discount_pct"] = np.where(
    menu_prices["is_promo"],
    (
        (
            menu_prices["regular_price_php"]
            - menu_prices["promo_price_php"]
        )
        / menu_prices["regular_price_php"]
    ) * 100,
    np.nan
)

menu_prices[
    [
        "brand",
        "product",
        "regular_price_php",
        "promo_price_php",
        "discount_php",
        "discount_pct"
    ]
].query("promo_price_php.notna()").head(10)

,brand,product,regular_price_php,promo_price_php,discount_php,discount_pct
0,"But First, Coffee",Spanish Latte,150,120.0,30.0,20.0
1,"But First, Coffee",Caramel Macchiato,150,120.0,30.0,20.0
2,"But First, Coffee",Matcha Latte,130,104.0,26.0,20.0
3,"But First, Coffee",Vietnamese,75,60.0,15.0,20.0
4,"But First, Coffee",Vietnamese,75,60.0,15.0,20.0
5,"But First, Coffee",Spanish Latte,130,104.0,26.0,20.0
6,"But First, Coffee",Caramel Macchiato,130,104.0,26.0,20.0
7,"But First, Coffee",Matcha Latte,130,104.0,26.0,20.0
8,"But First, Coffee",Choco Cookie Latte,180,144.0,36.0,20.0
31,"But First, Coffee",Cloud Nine Brew 16oz Iced,119,95.2,23.8,20.0


In [18]:
promo_issues = menu_prices[
    menu_prices["is_promo"]
    & (
        menu_prices["promo_price_php"]
        >= menu_prices["regular_price_php"]
    )
]

print(f"Potential promotional pricing issues: {len(promo_issues)}")

promo_issues[
    [
        "brand",
        "branch",
        "product",
        "regular_price_php",
        "promo_price_php"
    ]
]

Potential promotional pricing issues: 0


,brand,branch,product,regular_price_php,promo_price_php


In [19]:
promo_summary = (
    menu_prices
    .groupby("brand")
    .agg(
        total_products=("product", "count"),
        promo_products=("is_promo", "sum"),
        average_discount_pct=("discount_pct", "mean")
    )
    .reset_index()
)

promo_summary["promo_share_pct"] = (
    promo_summary["promo_products"]
    / promo_summary["total_products"]
    * 100
)

promo_summary[
    [
        "brand",
        "total_products",
        "promo_products",
        "promo_share_pct",
        "average_discount_pct"
    ]
].round(2)

,brand,total_products,promo_products,promo_share_pct,average_discount_pct
0,"But First, Coffee",50,45,90.0,20.0
1,Starbucks,50,0,0.0,NaN
2,The Coffee Bean & Tea Leaf,50,0,0.0,NaN


In [20]:
menu_prices["category_original"] = menu_prices["category"]

print(
    f"Original categories: "
    f"{menu_prices['category_original'].nunique()}"
)

Original categories: 31


In [21]:
category_mapping = {
    # Coffee
    "Espresso": "Coffee",
    "Brewed": "Coffee",
    "Latte": "Coffee",
    "Cold Brew": "Coffee",

    # Blended beverages
    "Frappuccino": "Blended Beverage",
    "Blended": "Blended Beverage",
    "Ice Blended Coffee": "Blended Beverage",
    "Ice Blended Coffee-Free": "Blended Beverage",

    # Matcha
    "Matcha": "Matcha",

    # Tea
    "Tea Latte": "Tea",

    # Non-coffee / specialty beverages
    "Chocolate": "Non-Coffee Beverage",
    "Non-coffee": "Non-Coffee Beverage",
    "Refreshers": "Non-Coffee Beverage",
    "Ube": "Non-Coffee Beverage",
    "Auro": "Non-Coffee Beverage",
    "Milky": "Non-Coffee Beverage",

    # Brand-specific beverage lines
    "Cloud Foam": "Specialty Beverage",
    "Signature": "Specialty Beverage",
    "Specialty": "Specialty Beverage",
    "Featured": "Specialty Beverage",
    "Limited": "Specialty Beverage",
    "Greater Cup": "Specialty Beverage",
    "Sip N Save": "Specialty Beverage",

    # Food
    "Pastry": "Food",
    "Bakery": "Food",
    "Cake": "Food",
    "Food": "Food",
    "Pasta": "Food",
    "Snack": "Food",

    # Add-ons
    "Add-on": "Add-on"
}

menu_prices["category_standard"] = (
    menu_prices["category_original"]
    .map(category_mapping)
)

In [22]:
unmapped_categories = (
    menu_prices.loc[
        menu_prices["category_standard"].isna(),
        "category_original"
    ]
    .drop_duplicates()
    .sort_values()
)

print(f"Unmapped categories: {len(unmapped_categories)}")

for category in unmapped_categories:
    print(f"- {category}")

Unmapped categories: 1
- Mocha


In [23]:
standard_category_summary = (
    menu_prices
    .groupby(["brand", "category_standard"])
    .size()
    .reset_index(name="product_count")
    .sort_values(
        ["brand", "product_count"],
        ascending=[True, False]
    )
)

standard_category_summary

,brand,category_standard,product_count
1,"But First, Coffee",Coffee,18
5,"But First, Coffee",Specialty Beverage,12
4,"But First, Coffee",Non-Coffee Beverage,10
3,"But First, Coffee",Matcha,5
2,"But First, Coffee",Food,3
0,"But First, Coffee",Add-on,2
7,Starbucks,Coffee,14
6,Starbucks,Blended Beverage,13
8,Starbucks,Food,12
10,Starbucks,Non-Coffee Beverage,6


In [24]:
food_categories = {
    "Food"
}

addon_categories = {
    "Add-on"
}

menu_prices["product_class"] = np.select(
    [
        menu_prices["category_standard"].isin(food_categories),
        menu_prices["category_standard"].isin(addon_categories)
    ],
    [
        "Food",
        "Add-on"
    ],
    default="Beverage"
)

menu_prices["product_class"].value_counts()

product_class
Beverage    122
Food         26
Add-on        2
Name: count, dtype: int64

In [25]:
pd.crosstab(
    menu_prices["brand"],
    menu_prices["product_class"],
    margins=True
)

product_class,Add-on,Beverage,Food,All
brand,,,,
"But First, Coffee",2,45,3,50
Starbucks,0,38,12,50
The Coffee Bean & Tea Leaf,0,39,11,50
All,2,122,26,150


In [26]:
category_mapping["Mocha"] = "Coffee"

menu_prices["category_standard"] = (
    menu_prices["category_original"]
    .map(category_mapping)
)

unmapped_categories = (
    menu_prices.loc[
        menu_prices["category_standard"].isna(),
        "category_original"
    ]
    .drop_duplicates()
    .sort_values()
)

print(f"Unmapped categories: {len(unmapped_categories)}")

Unmapped categories: 0


In [27]:
menu_prices["product_class"] = np.select(
    [
        menu_prices["category_standard"].eq("Food"),
        menu_prices["category_standard"].eq("Add-on")
    ],
    [
        "Food",
        "Add-on"
    ],
    default="Beverage"
)

In [28]:
beverage_prices = (
    menu_prices[
        menu_prices["product_class"] == "Beverage"
    ]
    .copy()
)

beverage_summary = (
    beverage_prices
    .groupby("brand")
    .agg(
        observations=("regular_price_php", "count"),
        mean_price=("regular_price_php", "mean"),
        median_price=("regular_price_php", "median"),
        minimum_price=("regular_price_php", "min"),
        maximum_price=("regular_price_php", "max"),
        std_price=("regular_price_php", "std")
    )
    .round(2)
    .reset_index()
)

beverage_summary

,brand,observations,mean_price,median_price,minimum_price,maximum_price,std_price
0,"But First, Coffee",45,141.91,130.0,69,219,42.08
1,Starbucks,38,196.84,195.0,140,220,15.40
2,The Coffee Bean & Tea Leaf,39,223.97,225.0,150,270,26.39


In [29]:
beverage_category_counts = pd.crosstab(
    beverage_prices["brand"],
    beverage_prices["category_standard"]
)

beverage_category_counts

category_standard,Blended Beverage,Coffee,Matcha,Non-Coffee Beverage,Specialty Beverage,Tea
brand,,,,,,
"But First, Coffee",0,18,5,10,12,0
Starbucks,13,14,2,6,3,0
The Coffee Bean & Tea Leaf,8,20,1,2,0,8


In [30]:
brand_category_presence = (
    beverage_prices
    .groupby("category_standard")["brand"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="brands_present")
)

brand_category_presence

,category_standard,brands_present
0,Coffee,3
1,Matcha,3
2,Non-Coffee Beverage,3
3,Blended Beverage,2
4,Specialty Beverage,2
5,Tea,1


In [31]:
common_categories = (
    brand_category_presence.loc[
        brand_category_presence["brands_present"] == 3,
        "category_standard"
    ]
    .tolist()
)

print("Categories represented across all three brands:")

for category in common_categories:
    print(f"- {category}")

Categories represented across all three brands:
- Coffee
- Matcha
- Non-Coffee Beverage


In [32]:
comparable_category_prices = (
    beverage_prices[
        beverage_prices["category_standard"].isin(common_categories)
    ]
    .copy()
)

pd.crosstab(
    comparable_category_prices["brand"],
    comparable_category_prices["category_standard"],
    margins=True
)

category_standard,Coffee,Matcha,Non-Coffee Beverage,All
brand,,,,
"But First, Coffee",18,5,10,33
Starbucks,14,2,6,22
The Coffee Bean & Tea Leaf,20,1,2,23
All,52,8,18,78


In [33]:
comparable_category_products = (
    comparable_category_prices[
        [
            "brand",
            "category_standard",
            "product",
            "regular_price_php"
        ]
    ]
    .sort_values(
        ["category_standard", "brand", "product"]
    )
)

comparable_category_products.to_string(index=False)

'                     brand   category_standard                                     product  regular_price_php\n         But First, Coffee              Coffee                                   Americano                130\n         But First, Coffee              Coffee                                      Brewed                 85\n         But First, Coffee              Coffee                                Butterscotch                150\n         But First, Coffee              Coffee                                  Cafe Latte                150\n         But First, Coffee              Coffee                           Caramel Macchiato                150\n         But First, Coffee              Coffee                           Caramel Macchiato                130\n         But First, Coffee              Coffee                           Caramel Macchiato                130\n         But First, Coffee              Coffee                                    Hazelnut                115\n

In [34]:
bfc_promos = (
    menu_prices[
        (menu_prices["brand"] == "But First, Coffee")
        & (menu_prices["is_promo"])
    ]
    .copy()
)

bfc_promos["discount_pct"].describe()

count    4.500000e+01
mean     2.000000e+01
std      1.514882e-15
min      2.000000e+01
25%      2.000000e+01
50%      2.000000e+01
75%      2.000000e+01
max      2.000000e+01
Name: discount_pct, dtype: float64

In [35]:
bfc_promos["discount_pct"].round(2).value_counts().sort_index()

discount_pct
20.0    45
Name: count, dtype: int64

In [36]:
def assign_product_family(product):
    product_lower = product.lower().strip()

    # Americano
    if "americano" in product_lower:
        return "Americano"

    # Caramel Macchiato
    if "caramel macchiato" in product_lower:
        return "Caramel Macchiato"

    # Matcha Latte
    if (
        "matcha latte" in product_lower
        or "pure matcha latte" in product_lower
        or "matcha tea latte" in product_lower
    ):
        return "Matcha Latte"

    # Mocha family
    if (
        "white mocha" in product_lower
        or "caffe mocha" in product_lower
        or "café mocha" in product_lower
        or "mocha latte" in product_lower
        or "original mocha" in product_lower
    ):
        return "Mocha"

    # Plain café latte
    if (
        product_lower in {
            "cafe latte",
            "café latte",
            "caffe latte",
            "iced café latte",
            "iced cafe latte"
        }
    ):
        return "Cafe Latte"

    return np.nan


menu_prices["comparable_product_family"] = (
    menu_prices["product"]
    .apply(assign_product_family)
)

In [37]:
matched_products = (
    menu_prices[
        menu_prices["comparable_product_family"].notna()
    ]
    .copy()
)

matched_products[
    [
        "brand",
        "branch",
        "product",
        "comparable_product_family",
        "regular_price_php"
    ]
].sort_values(
    ["comparable_product_family", "brand"]
)

,brand,branch,product,comparable_product_family,regular_price_php
35,"But First, Coffee",Malingap Street,Americano,Americano,130
61,"But First, Coffee",Malingap Street,Salted Caramel Americano,Americano,200
14,Starbucks,6750 Building,Caffe Americano,Americano,175
19,Starbucks,SM City Manila,Caffe Americano,Americano,175
131,The Coffee Bean & Tea Leaf,SM Seaside,Americano,Americano,185
132,The Coffee Bean & Tea Leaf,SM Seaside,Iced Americano,Americano,185
37,"But First, Coffee",Malingap Street,Cafe Latte,Cafe Latte,150
15,Starbucks,6750 Building,Caffe Latte,Cafe Latte,185
20,Starbucks,SM City Manila,Caffe Latte,Cafe Latte,185
121,The Coffee Bean & Tea Leaf,SM Seaside,Iced Café Latte,Cafe Latte,200


In [38]:
matched_coverage = pd.crosstab(
    matched_products["comparable_product_family"],
    matched_products["brand"]
)

matched_coverage

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,2,2,2
Cafe Latte,1,2,2
Caramel Macchiato,3,2,4
Matcha Latte,3,2,2
Mocha,1,3,5


In [39]:
three_brand_families = (
    matched_products
    .groupby("comparable_product_family")["brand"]
    .nunique()
)

three_brand_families = (
    three_brand_families[
        three_brand_families == 3
    ]
    .index
    .tolist()
)

print("Three-brand comparable product families:")

for family in three_brand_families:
    print(f"- {family}")

Three-brand comparable product families:
- Americano
- Cafe Latte
- Caramel Macchiato
- Matcha Latte
- Mocha


In [40]:
comparable_prices = (
    matched_products[
        matched_products[
            "comparable_product_family"
        ].isin(three_brand_families)
    ]
    .copy()
)

comparison_counts = pd.crosstab(
    comparable_prices["brand"],
    comparable_prices["comparable_product_family"],
    margins=True
)

comparison_counts

comparable_product_family,Americano,Cafe Latte,Caramel Macchiato,Matcha Latte,Mocha,All
brand,,,,,,
"But First, Coffee",2,1,3,3,1,10
Starbucks,2,2,2,2,3,11
The Coffee Bean & Tea Leaf,2,2,4,2,5,15
All,6,5,9,7,9,36


In [41]:
family_price_summary = (
    comparable_prices
    .groupby(
        ["comparable_product_family", "brand"]
    )
    .agg(
        observations=("regular_price_php", "count"),
        mean_price=("regular_price_php", "mean"),
        median_price=("regular_price_php", "median"),
        minimum_price=("regular_price_php", "min"),
        maximum_price=("regular_price_php", "max")
    )
    .round(2)
    .reset_index()
)

family_price_summary

,comparable_product_family,brand,observations,mean_price,median_price,minimum_price,maximum_price
0,Americano,"But First, Coffee",2,165.00,165.0,130,200
1,Americano,Starbucks,2,175.00,175.0,175,175
2,Americano,The Coffee Bean & Tea Leaf,2,185.00,185.0,185,185
3,Cafe Latte,"But First, Coffee",1,150.00,150.0,150,150
4,Cafe Latte,Starbucks,2,185.00,185.0,185,185
5,Cafe Latte,The Coffee Bean & Tea Leaf,2,187.50,187.5,175,200
6,Caramel Macchiato,"But First, Coffee",3,136.67,130.0,130,150
7,Caramel Macchiato,Starbucks,2,210.00,210.0,210,210
8,Caramel Macchiato,The Coffee Bean & Tea Leaf,4,245.00,245.0,245,245
9,Matcha Latte,"But First, Coffee",3,130.00,130.0,130,130


In [42]:
price_matrix = (
    comparable_prices
    .pivot_table(
        index="comparable_product_family",
        columns="brand",
        values="regular_price_php",
        aggfunc="median"
    )
    .round(2)
)

price_matrix

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,165.0,175.0,185.0
Cafe Latte,150.0,185.0,187.5
Caramel Macchiato,130.0,210.0,245.0
Matcha Latte,130.0,190.0,225.0
Mocha,150.0,200.0,220.0


In [43]:
price_index = price_matrix.copy()

price_index["BFC_vs_Starbucks_Index"] = (
    price_index["But First, Coffee"]
    / price_index["Starbucks"]
    * 100
)

price_index["BFC_vs_CBTL_Index"] = (
    price_index["But First, Coffee"]
    / price_index["The Coffee Bean & Tea Leaf"]
    * 100
)

price_index[
    [
        "BFC_vs_Starbucks_Index",
        "BFC_vs_CBTL_Index"
    ]
].round(1)

brand,BFC_vs_Starbucks_Index,BFC_vs_CBTL_Index
comparable_product_family,,
Americano,94.3,89.2
Cafe Latte,81.1,80.0
Caramel Macchiato,61.9,53.1
Matcha Latte,68.4,57.8
Mocha,75.0,68.2


In [44]:
price_gap = price_matrix.copy()

price_gap["BFC_vs_Starbucks_Gap_Pct"] = (
    (
        price_gap["But First, Coffee"]
        / price_gap["Starbucks"]
    ) - 1
) * 100

price_gap["BFC_vs_CBTL_Gap_Pct"] = (
    (
        price_gap["But First, Coffee"]
        / price_gap["The Coffee Bean & Tea Leaf"]
    ) - 1
) * 100

price_gap[
    [
        "BFC_vs_Starbucks_Gap_Pct",
        "BFC_vs_CBTL_Gap_Pct"
    ]
].round(1)

brand,BFC_vs_Starbucks_Gap_Pct,BFC_vs_CBTL_Gap_Pct
comparable_product_family,,
Americano,-5.7,-10.8
Cafe Latte,-18.9,-20.0
Caramel Macchiato,-38.1,-46.9
Matcha Latte,-31.6,-42.2
Mocha,-25.0,-31.8


In [45]:
bfc_comparable = (
    comparable_prices[
        comparable_prices["brand"] == "But First, Coffee"
    ]
    .copy()
)

bfc_comparable_summary = (
    bfc_comparable
    .groupby("comparable_product_family")
    .agg(
        regular_median=("regular_price_php", "median"),
        promo_median=("promo_price_php", "median")
    )
)

bfc_comparable_summary["promo_gap_pct"] = (
    (
        bfc_comparable_summary["promo_median"]
        / bfc_comparable_summary["regular_median"]
    ) - 1
) * 100

bfc_comparable_summary.round(2)

,regular_median,promo_median,promo_gap_pct
comparable_product_family,,,
Americano,165.0,132.0,-20.00
Cafe Latte,150.0,120.0,-20.00
Caramel Macchiato,130.0,112.0,-13.85
Matcha Latte,130.0,104.0,-20.00
Mocha,150.0,120.0,-20.00


In [46]:
def assign_product_family(product):
    product_lower = product.lower().strip()

    # Plain Americano only
    americano_names = {
        "americano",
        "caffe americano",
        "café americano",
        "iced americano"
    }

    if product_lower in americano_names:
        return "Americano"

    # Plain Cafe Latte
    cafe_latte_names = {
        "cafe latte",
        "café latte",
        "caffe latte",
        "iced cafe latte",
        "iced café latte"
    }

    if product_lower in cafe_latte_names:
        return "Cafe Latte"

    # Caramel Macchiato
    caramel_macchiato_names = {
        "caramel macchiato",
        "iced caramel macchiato"
    }

    if product_lower in caramel_macchiato_names:
        return "Caramel Macchiato"

    # Matcha Latte
    matcha_latte_names = {
        "matcha latte",
        "pure matcha latte",
        "iced matcha tea latte"
    }

    if product_lower in matcha_latte_names:
        return "Matcha Latte"

    # Regular Mocha drinks
    mocha_names = {
        "white mocha",
        "caffe mocha",
        "café mocha",
        "mocha latte",
        "iced mocha latte",
        "the original mocha"
    }

    if product_lower in mocha_names:
        return "Mocha"

    return np.nan


menu_prices["comparable_product_family"] = (
    menu_prices["product"]
    .apply(assign_product_family)
)

In [47]:
matched_products = (
    menu_prices[
        menu_prices["comparable_product_family"].notna()
    ]
    .copy()
)

matched_coverage = pd.crosstab(
    matched_products["comparable_product_family"],
    matched_products["brand"]
)

matched_coverage

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,1,2,2
Cafe Latte,1,2,2
Caramel Macchiato,3,2,4
Matcha Latte,3,2,2
Mocha,1,2,5


In [48]:
three_brand_families = (
    matched_products
    .groupby("comparable_product_family")["brand"]
    .nunique()
)

three_brand_families = (
    three_brand_families[
        three_brand_families == 3
    ]
    .index
    .tolist()
)

comparable_prices = (
    matched_products[
        matched_products["comparable_product_family"]
        .isin(three_brand_families)
    ]
    .copy()
)

print(three_brand_families)

['Americano', 'Cafe Latte', 'Caramel Macchiato', 'Matcha Latte', 'Mocha']


In [49]:
price_matrix = (
    comparable_prices
    .pivot_table(
        index="comparable_product_family",
        columns="brand",
        values="regular_price_php",
        aggfunc="median"
    )
    .round(2)
)

price_matrix

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,130.0,175.0,185.0
Cafe Latte,150.0,185.0,187.5
Caramel Macchiato,130.0,210.0,245.0
Matcha Latte,130.0,190.0,225.0
Mocha,150.0,205.0,220.0


In [50]:
price_gap = price_matrix.copy()

price_gap["BFC_vs_Starbucks_Gap_Pct"] = (
    (
        price_gap["But First, Coffee"]
        / price_gap["Starbucks"]
    ) - 1
) * 100

price_gap["BFC_vs_CBTL_Gap_Pct"] = (
    (
        price_gap["But First, Coffee"]
        / price_gap["The Coffee Bean & Tea Leaf"]
    ) - 1
) * 100

price_gap[
    [
        "BFC_vs_Starbucks_Gap_Pct",
        "BFC_vs_CBTL_Gap_Pct"
    ]
].round(1)

brand,BFC_vs_Starbucks_Gap_Pct,BFC_vs_CBTL_Gap_Pct
comparable_product_family,,
Americano,-25.7,-29.7
Cafe Latte,-18.9,-20.0
Caramel Macchiato,-38.1,-46.9
Matcha Latte,-31.6,-42.2
Mocha,-26.8,-31.8


In [51]:
matched_family_prices = (
    comparable_prices
    .groupby(
        [
            "brand",
            "comparable_product_family"
        ],
        as_index=False
    )
    .agg(
        representative_price_php=(
            "regular_price_php",
            "median"
        ),
        source_observations=(
            "regular_price_php",
            "count"
        )
    )
)

matched_family_prices

,brand,comparable_product_family,representative_price_php,source_observations
0,"But First, Coffee",Americano,130.0,1
1,"But First, Coffee",Cafe Latte,150.0,1
2,"But First, Coffee",Caramel Macchiato,130.0,3
3,"But First, Coffee",Matcha Latte,130.0,3
4,"But First, Coffee",Mocha,150.0,1
5,Starbucks,Americano,175.0,2
6,Starbucks,Cafe Latte,185.0,2
7,Starbucks,Caramel Macchiato,210.0,2
8,Starbucks,Matcha Latte,190.0,2
9,Starbucks,Mocha,205.0,2


In [52]:
print(
    "Representative observations:",
    len(matched_family_prices)
)

pd.crosstab(
    matched_family_prices["comparable_product_family"],
    matched_family_prices["brand"]
)

Representative observations: 15


brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,1,1,1
Cafe Latte,1,1,1
Caramel Macchiato,1,1,1
Matcha Latte,1,1,1
Mocha,1,1,1


In [53]:
bfc_promo_summary = (
    menu_prices[
        (menu_prices["brand"] == "But First, Coffee")
        & (menu_prices["is_promo"])
    ]
    .groupby("comparable_product_family", dropna=False)
    .agg(
        observations=("discount_pct", "count"),
        mean_discount_pct=("discount_pct", "mean"),
        median_discount_pct=("discount_pct", "median"),
        min_discount_pct=("discount_pct", "min"),
        max_discount_pct=("discount_pct", "max")
    )
    .round(2)
)

bfc_promo_summary

,observations,mean_discount_pct,median_discount_pct,min_discount_pct,max_discount_pct
comparable_product_family,,,,,
Americano,1,20.0,20.0,20.0,20.0
Cafe Latte,1,20.0,20.0,20.0,20.0
Caramel Macchiato,2,20.0,20.0,20.0,20.0
Matcha Latte,2,20.0,20.0,20.0,20.0
Mocha,1,20.0,20.0,20.0,20.0
NaN,38,20.0,20.0,20.0,20.0


## Menu Pricing Cleaning Summary

The menu-pricing dataset contains 150 observations, with 50 observations collected for each competitor brand.

Product composition differs across brands. But First, Coffee is more heavily represented by beverage products, while Starbucks and The Coffee Bean & Tea Leaf contain a larger proportion of food observations. Therefore, the complete 150-product dataset will be retained for descriptive menu analysis but will not be treated as a fully comparable sample for inferential pricing analysis.

Original menu categories were preserved and a standardized analytical category variable was created.

For stronger cross-brand pricing comparisons, five comparable product families were identified:

- Americano
- Cafe Latte
- Caramel Macchiato
- Matcha Latte
- Mocha

Repeated observations of the same product family within a brand will be summarized using the median price so that brands with more sampled branches do not receive disproportionate statistical weight.

The resulting matched dataset contains one representative price for each brand within each comparable product family.

Promotional pricing is analyzed separately from regular pricing. In the collected snapshot, 45 of 50 But First, Coffee menu observations contained a promotional price, and all observed promotional prices represented a 20% reduction from their corresponding regular prices.

Missing promotional prices are retained as missing values because they indicate that no promotional price was observed rather than a zero-priced product.

# Customer Review Cleaning

The customer-review dataset contains written customer feedback collected across multiple branches of But First, Coffee, Starbucks, and The Coffee Bean & Tea Leaf.

The cleaning process will preserve the original review text and raw theme labels while creating standardized analytical variables.

Because a single customer review may contain multiple experience themes, themes will be treated as multi-label variables rather than forcing each review into one mutually exclusive category.

The review dataset will primarily support thematic customer-experience analysis. Branch-level aggregate ratings will not be interpreted as individual customer ratings.

In [54]:
reviews["review_text_original"] = reviews["review_text"]
reviews["theme_original"] = reviews["initial_theme_hint"]

print("Original review text and theme labels preserved.")

Original review text and theme labels preserved.


In [55]:
reviews["review_text_clean"] = (
    reviews["review_text_original"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

reviews[
    [
        "review_text_original",
        "review_text_clean"
    ]
].head()

,review_text_original,review_text_clean
0,There are no straws provided,There are no straws provided
1,Lacks espresso. Gatas lang yung nalalasahan,Lacks espresso. Gatas lang yung nalalasahan
2,I love but first coffee i always buy coffee there but this branch not good or standard of how ma...,I love but first coffee i always buy coffee there but this branch not good or standard of how ma...
3,"The cap was not properly sealed, and since it was made of paper, the spilled liquid was absorbed...","The cap was not properly sealed, and since it was made of paper, the spilled liquid was absorbed..."
4,lasang tubig na may konting kape. Sayang pera sa inyo,lasang tubig na may konting kape. Sayang pera sa inyo


In [56]:
blank_reviews = (
    reviews["review_text_clean"].isna()
    | reviews["review_text_clean"].eq("")
)

print(f"Blank review texts: {blank_reviews.sum()}")
print(
    f"Usable review texts: "
    f"{(~blank_reviews).sum()}"
)

Blank review texts: 0
Usable review texts: 282


In [57]:
reviews["theme_list"] = (
    reviews["theme_original"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.split("/")
)

reviews[
    [
        "theme_original",
        "theme_list"
    ]
].head(15)

,theme_original,theme_list
0,packaging/accessories,"[packaging, accessories]"
1,taste/coffee strength,"[taste, coffee strength]"
2,consistency,[consistency]
3,packaging,[packaging]
4,taste/value,"[taste, value]"
5,order accuracy,[order accuracy]
6,customization/service,"[customization, service]"
7,taste/missing add-on,"[taste, missing add-on]"
8,consistency/taste,"[consistency, taste]"
9,order accuracy/coffee strength,"[order accuracy, coffee strength]"


In [58]:
atomic_themes = sorted({
    theme.strip()
    for themes in reviews["theme_list"].dropna()
    for theme in themes
    if theme.strip()
})

print(
    f"Number of atomic themes: "
    f"{len(atomic_themes)}"
)

print("\nAtomic themes:")

for theme in atomic_themes:
    print(f"- {theme}")

Number of atomic themes: 27

Atomic themes:
- accessories
- add-on
- availability
- coffee strength
- consistency
- customization
- delivery
- food
- food quality
- loyalty
- menu accuracy
- missing add-on
- missing item
- occasion
- order accuracy
- order identification
- packaging
- portion
- portion consistency
- positive
- quality
- service
- size accuracy
- speed
- taste
- temperature
- value


In [59]:
atomic_theme_counts = []

for theme in atomic_themes:
    count = reviews["theme_list"].apply(
        lambda themes:
        theme in themes
        if isinstance(themes, list)
        else False
    ).sum()

    atomic_theme_counts.append({
        "atomic_theme": theme,
        "review_count": count
    })

atomic_theme_counts = (
    pd.DataFrame(atomic_theme_counts)
    .sort_values(
        "review_count",
        ascending=False
    )
    .reset_index(drop=True)
)

atomic_theme_counts

,atomic_theme,review_count
0,taste,87
1,packaging,55
2,order accuracy,40
3,customization,37
4,consistency,27
5,portion,22
6,missing item,22
7,value,20
8,add-on,13
9,service,13


In [60]:
theme_brand_records = []

for _, row in reviews.iterrows():
    themes = row["theme_list"]

    if isinstance(themes, list):
        for theme in themes:
            theme_brand_records.append({
                "brand": row["brand"],
                "atomic_theme": theme.strip()
            })

theme_brand_long = pd.DataFrame(
    theme_brand_records
)

theme_brand_counts = (
    theme_brand_long
    .groupby(
        ["brand", "atomic_theme"]
    )
    .size()
    .reset_index(name="review_mentions")
)

theme_brand_counts.head(20)

,brand,atomic_theme,review_mentions
0,"But First, Coffee",accessories,1
1,"But First, Coffee",add-on,4
2,"But First, Coffee",coffee strength,2
3,"But First, Coffee",consistency,14
4,"But First, Coffee",customization,9
5,"But First, Coffee",delivery,1
6,"But First, Coffee",food quality,1
7,"But First, Coffee",loyalty,3
8,"But First, Coffee",missing add-on,1
9,"But First, Coffee",missing item,4


In [61]:
brand_review_totals = (
    reviews
    .groupby("brand")
    .size()
    .rename("total_reviews")
    .reset_index()
)

theme_brand_prevalence = (
    theme_brand_counts
    .merge(
        brand_review_totals,
        on="brand",
        how="left"
    )
)

theme_brand_prevalence["prevalence_pct"] = (
    theme_brand_prevalence["review_mentions"]
    / theme_brand_prevalence["total_reviews"]
    * 100
)

theme_brand_prevalence = (
    theme_brand_prevalence
    .sort_values(
        ["brand", "prevalence_pct"],
        ascending=[True, False]
    )
)

theme_brand_prevalence.head(30)

,brand,atomic_theme,review_mentions,total_reviews,prevalence_pct
18,"But First, Coffee",taste,49,104,47.115385
12,"But First, Coffee",packaging,20,104,19.230769
19,"But First, Coffee",value,16,104,15.384615
3,"But First, Coffee",consistency,14,104,13.461538
11,"But First, Coffee",order accuracy,11,104,10.576923
13,"But First, Coffee",portion,10,104,9.615385
4,"But First, Coffee",customization,9,104,8.653846
16,"But First, Coffee",service,6,104,5.769231
1,"But First, Coffee",add-on,4,104,3.846154
9,"But First, Coffee",missing item,4,104,3.846154


In [62]:
print("Review date values:")
print(
    reviews["review_date"]
    .value_counts(dropna=False)
    .head(20)
)

print(
    "\nMissing review dates:",
    reviews["review_date"].isna().sum()
)

Review date values:
review_date
NaN           249
2026-04-20      2
2026-04-21      1
2026-04-07      1
2026-08-22      1
2026-08-19      1
2026-05-31      1
2025-10-25      1
2026-05-04      1
2026-05-29      1
2026-03-25      1
2026-03-18      1
2026-05-28      1
2026-03-28      1
2026-08-16      1
2026-09-16      1
2026-07-06      1
2026-09-04      1
2026-06-15      1
2026-03-19      1
Name: count, dtype: int64

Missing review dates: 249


In [63]:
reviews["review_date_parsed"] = pd.to_datetime(
    reviews["review_date"],
    errors="coerce"
)

print(
    "Successfully parsed dates:",
    reviews["review_date_parsed"]
    .notna()
    .sum()
)

print(
    "Unparsed or missing dates:",
    reviews["review_date_parsed"]
    .isna()
    .sum()
)

Successfully parsed dates: 33
Unparsed or missing dates: 249


In [64]:
rating_volume_audit = (
    reviews["rating_volume"]
    .value_counts(dropna=False)
    .rename_axis("rating_volume")
    .reset_index(name="rows")
)

rating_volume_audit

,rating_volume,rows
0,500+,95
1,100+,71
2,2000+,47
3,1000+,41
4,3000+,12
5,36,5
6,55,5
7,4000+,4
8,35,2


In [65]:
atomic_theme_mapping = {
    "accessories": "packaging_accessories",
    "add-on": "add_on",
    "availability": "availability",
    "coffee strength": "coffee_strength",
    "consistency": "consistency",
    "customization": "customization",
    "delivery": "delivery",

    "food": "food_quality",
    "food quality": "food_quality",

    "loyalty": "loyalty",

    "menu accuracy": "order_accuracy",
    "missing item": "missing_item",
    "missing add-on": "missing_add_on",

    "occasion": "occasion",

    "order accuracy": "order_accuracy",
    "order identification": "order_identification",

    "packaging": "packaging",

    "portion": "portion",
    "portion consistency": "portion_consistency",

    "positive": "positive",

    "quality": "quality",

    "service": "service",

    "size accuracy": "size_accuracy",

    "speed": "speed",
    "taste": "taste",
    "temperature": "temperature",
    "value": "value"
}

In [66]:
reviews["theme_standard_list"] = (
    reviews["theme_list"]
    .apply(
        lambda themes: [
            atomic_theme_mapping.get(
                theme.strip(),
                theme.strip().replace(" ", "_")
            )
            for theme in themes
        ]
        if isinstance(themes, list)
        else []
    )
)

reviews[
    [
        "theme_original",
        "theme_list",
        "theme_standard_list"
    ]
].head(15)

,theme_original,theme_list,theme_standard_list
0,packaging/accessories,"[packaging, accessories]","[packaging, packaging_accessories]"
1,taste/coffee strength,"[taste, coffee strength]","[taste, coffee_strength]"
2,consistency,[consistency],[consistency]
3,packaging,[packaging],[packaging]
4,taste/value,"[taste, value]","[taste, value]"
5,order accuracy,[order accuracy],[order_accuracy]
6,customization/service,"[customization, service]","[customization, service]"
7,taste/missing add-on,"[taste, missing add-on]","[taste, missing_add_on]"
8,consistency/taste,"[consistency, taste]","[consistency, taste]"
9,order accuracy/coffee strength,"[order accuracy, coffee strength]","[order_accuracy, coffee_strength]"


In [67]:
standard_atomic_themes = sorted({
    theme
    for themes in reviews["theme_standard_list"]
    for theme in themes
})

print(
    f"Standardized atomic themes: "
    f"{len(standard_atomic_themes)}"
)

for theme in standard_atomic_themes:
    print(f"- {theme}")

Standardized atomic themes: 25
- add_on
- availability
- coffee_strength
- consistency
- customization
- delivery
- food_quality
- loyalty
- missing_add_on
- missing_item
- occasion
- order_accuracy
- order_identification
- packaging
- packaging_accessories
- portion
- portion_consistency
- positive
- quality
- service
- size_accuracy
- speed
- taste
- temperature
- value


In [68]:
for theme in standard_atomic_themes:
    column_name = f"theme_{theme}"

    reviews[column_name] = (
        reviews["theme_standard_list"]
        .apply(
            lambda themes:
            int(theme in themes)
        )
    )

theme_columns = [
    column
    for column in reviews.columns
    if column.startswith("theme_")
    and column not in {
        "theme_original",
        "theme_list",
        "theme_standard_list"
    }
]

print(
    f"Theme indicator columns created: "
    f"{len(theme_columns)}"
)

theme_columns

Theme indicator columns created: 25


['theme_add_on',
 'theme_availability',
 'theme_coffee_strength',
 'theme_consistency',
 'theme_customization',
 'theme_delivery',
 'theme_food_quality',
 'theme_loyalty',
 'theme_missing_add_on',
 'theme_missing_item',
 'theme_occasion',
 'theme_order_accuracy',
 'theme_order_identification',
 'theme_packaging',
 'theme_packaging_accessories',
 'theme_portion',
 'theme_portion_consistency',
 'theme_positive',
 'theme_quality',
 'theme_service',
 'theme_size_accuracy',
 'theme_speed',
 'theme_taste',
 'theme_temperature',
 'theme_value']

In [69]:
reviews["theme_count"] = (
    reviews[theme_columns]
    .sum(axis=1)
)

reviews["theme_count"].value_counts().sort_index()

theme_count
1    174
2    106
3      2
Name: count, dtype: int64

In [70]:
print(
    "Reviews with multiple themes:",
    (reviews["theme_count"] > 1).sum()
)

print(
    "Maximum themes in one review:",
    reviews["theme_count"].max()
)

Reviews with multiple themes: 108
Maximum themes in one review: 3


In [71]:
management_dimensions = {
    "product_quality": {
        "taste",
        "coffee_strength",
        "food_quality",
        "quality",
        "temperature"
    },

    "order_execution": {
        "order_accuracy",
        "missing_item",
        "missing_add_on",
        "size_accuracy"
    },

    "customization_addons": {
        "customization",
        "add_on"
    },

    "packaging": {
        "packaging",
        "packaging_accessories",
        "order_identification"
    },

    "value_portion": {
        "value",
        "portion",
        "portion_consistency"
    },

    "service_delivery": {
        "service",
        "speed",
        "delivery"
    },

    "availability_consistency": {
        "availability",
        "consistency"
    },

    "loyalty_experience": {
        "loyalty",
        "occasion",
        "positive"
    }
}

In [72]:
for dimension, themes in management_dimensions.items():

    relevant_columns = [
        f"theme_{theme}"
        for theme in themes
        if f"theme_{theme}" in reviews.columns
    ]

    reviews[f"dimension_{dimension}"] = (
        reviews[relevant_columns]
        .max(axis=1)
    )

dimension_columns = [
    column
    for column in reviews.columns
    if column.startswith("dimension_")
]

dimension_columns

['dimension_product_quality',
 'dimension_order_execution',
 'dimension_customization_addons',
 'dimension_packaging',
 'dimension_value_portion',
 'dimension_service_delivery',
 'dimension_availability_consistency',
 'dimension_loyalty_experience']

In [73]:
dimension_prevalence = (
    reviews
    .groupby("brand")[dimension_columns]
    .mean()
    .mul(100)
    .round(2)
    .T
)

dimension_prevalence

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
dimension_product_quality,50.00,20.37,37.14
dimension_order_execution,15.38,32.41,27.14
dimension_customization_addons,12.50,28.70,5.71
dimension_packaging,19.23,16.67,24.29
dimension_value_portion,19.23,6.48,11.43
dimension_service_delivery,7.69,8.33,2.86
dimension_availability_consistency,13.46,7.41,12.86
dimension_loyalty_experience,3.85,0.00,7.14


In [74]:
reviews["rating_volume_original"] = (
    reviews["rating_volume"]
)

reviews["rating_volume_is_lower_bound"] = (
    reviews["rating_volume_original"]
    .astype(str)
    .str.contains(r"\+", regex=True)
)

reviews["rating_volume_min"] = (
    reviews["rating_volume_original"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(int)
)

reviews[
    [
        "rating_volume_original",
        "rating_volume_min",
        "rating_volume_is_lower_bound"
    ]
].drop_duplicates().sort_values(
    "rating_volume_min"
)

,rating_volume_original,rating_volume_min,rating_volume_is_lower_bound
48,35,35,False
129,36,36,False
248,55,55,False
12,100+,100,True
0,500+,500,True
6,1000+,1000,True
56,2000+,2000,True
22,3000+,3000,True
63,4000+,4000,True


In [75]:
aggregate_rating_summary = (
    reviews
    .groupby("brand")
    .agg(
        observations=("aggregate_rating", "count"),
        minimum_rating=("aggregate_rating", "min"),
        maximum_rating=("aggregate_rating", "max"),
        mean_displayed_rating=("aggregate_rating", "mean"),
        median_displayed_rating=("aggregate_rating", "median")
    )
    .round(2)
    .reset_index()
)

aggregate_rating_summary

,brand,observations,minimum_rating,maximum_rating,mean_displayed_rating,median_displayed_rating
0,"But First, Coffee",104,4.9,5.0,5.00,5.0
1,Starbucks,108,4.9,5.0,4.99,5.0
2,The Coffee Bean & Tea Leaf,70,4.8,5.0,4.98,5.0


In [76]:
branch_rating_records = (
    reviews[
        [
            "brand",
            "branch",
            "platform",
            "aggregate_rating",
            "rating_volume_original",
            "rating_volume_min",
            "rating_volume_is_lower_bound"
        ]
    ]
    .drop_duplicates()
)

print(
    "Unique branch/platform rating records:",
    len(branch_rating_records)
)

branch_rating_records.groupby("brand").size()

Unique branch/platform rating records: 55


brand
But First, Coffee             19
Starbucks                     17
The Coffee Bean & Tea Leaf    19
dtype: int64

In [77]:
review_business_duplicates = (
    reviews[
        reviews.duplicated(
            subset=[
                "brand",
                "branch",
                "platform",
                "review_text_original"
            ],
            keep=False
        )
    ]
    .sort_values(
        ["brand", "branch", "review_text_original"]
    )
)

print(
    "Potential duplicate review rows:",
    len(review_business_duplicates)
)

review_business_duplicates[
    [
        "brand",
        "branch",
        "platform",
        "review_text_original"
    ]
].head(20)

Potential duplicate review rows: 0


,brand,branch,platform,review_text_original


In [78]:
menu_business_duplicates = (
    menu_prices[
        menu_prices.duplicated(
            subset=[
                "brand",
                "branch",
                "category_original",
                "product",
                "platform"
            ],
            keep=False
        )
    ]
    .sort_values(
        ["brand", "branch", "product"]
    )
)

print(
    "Potential duplicate menu rows:",
    len(menu_business_duplicates)
)

menu_business_duplicates[
    [
        "brand",
        "branch",
        "category_original",
        "product",
        "regular_price_php",
        "promo_price_php",
        "platform"
    ]
].head(30)

Potential duplicate menu rows: 0


,brand,branch,category_original,product,regular_price_php,promo_price_php,platform


In [79]:
individual_rating_summary = (
    individual_ratings
    .groupby("brand")
    .agg(
        observations=("rating", "count"),
        mean_rating=("rating", "mean"),
        median_rating=("rating", "median"),
        minimum_rating=("rating", "min"),
        maximum_rating=("rating", "max"),
        branches=("branch", "nunique")
    )
    .round(2)
    .reset_index()
)

individual_rating_summary

,brand,observations,mean_rating,median_rating,minimum_rating,maximum_rating,branches
0,"But First, Coffee",13,4.69,5.0,2,5,2


In [80]:
invalid_individual_ratings = (
    individual_ratings[
        individual_ratings["rating"].isna()
        | ~individual_ratings["rating"].between(1, 5)
    ]
)

print(
    "Invalid individual ratings:",
    len(invalid_individual_ratings)
)

Invalid individual ratings: 0


In [81]:
individual_rating_distribution = (
    individual_ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)

individual_rating_distribution["percentage"] = (
    individual_rating_distribution["count"]
    / individual_rating_distribution["count"].sum()
    * 100
)

individual_rating_distribution.round(2)

,rating,count,percentage
0,2,1,7.69
1,4,1,7.69
2,5,11,84.62


In [82]:
individual_rating_brand_coverage = (
    individual_ratings
    .groupby("brand")
    .agg(
        ratings=("rating", "count"),
        branches=("branch", "nunique")
    )
    .reset_index()
)

individual_rating_brand_coverage

,brand,ratings,branches
0,"But First, Coffee",13,2


In [83]:
required_brands = {
    "But First, Coffee",
    "Starbucks",
    "The Coffee Bean & Tea Leaf"
}

available_rating_brands = set(
    individual_ratings["brand"].unique()
)

missing_rating_brands = (
    required_brands
    - available_rating_brands
)

print(
    "Brands with individual ratings:",
    available_rating_brands
)

print(
    "Brands missing individual ratings:",
    missing_rating_brands
)

print(
    "Cross-brand rating hypothesis test possible:",
    len(missing_rating_brands) == 0
)

Brands with individual ratings: {'But First, Coffee'}
Brands missing individual ratings: {'Starbucks', 'The Coffee Bean & Tea Leaf'}
Cross-brand rating hypothesis test possible: False


In [84]:
RATING_HYPOTHESIS_TEST_ELIGIBLE = (
    len(missing_rating_brands) == 0
)

print(
    "RQ2 inferential testing eligible:",
    RATING_HYPOTHESIS_TEST_ELIGIBLE
)

RQ2 inferential testing eligible: False


In [85]:
cleaning_datasets = {
    "reviews": reviews,
    "menu_prices": menu_prices,
    "store_footprint": store_footprint,
    "channels": channels,
    "menu_breadth": menu_breadth,
    "official_menu": official_menu,
    "official_capabilities": official_capabilities,
    "individual_ratings": individual_ratings
}

missingness_records = []

for dataset_name, df in cleaning_datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            missingness_records.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": missing_count,
                "missing_pct": (
                    missing_count / len(df) * 100
                )
            })

missingness_summary = (
    pd.DataFrame(missingness_records)
    .sort_values(
        ["dataset", "missing_pct"],
        ascending=[True, False]
    )
)

missingness_summary.round(2)

,dataset,column,missing_count,missing_pct
5,menu_prices,comparable_product_family,116,77.33
2,menu_prices,promo_price_php,105,70.00
3,menu_prices,discount_php,105,70.00
4,menu_prices,discount_pct,105,70.00
0,reviews,review_date,249,88.30
1,reviews,review_date_parsed,249,88.30


In [86]:
final_row_counts = pd.DataFrame({
    "dataset": [
        "reviews",
        "menu_prices",
        "store_footprint",
        "channels",
        "menu_breadth",
        "official_menu",
        "official_capabilities",
        "individual_ratings"
    ],
    "raw_rows": [
        len(reviews_raw),
        len(menu_prices_raw),
        len(store_footprint_raw),
        len(channels_raw),
        len(menu_breadth_raw),
        len(official_menu_raw),
        len(official_capabilities_raw),
        len(individual_ratings_raw)
    ],
    "cleaned_rows": [
        len(reviews),
        len(menu_prices),
        len(store_footprint),
        len(channels),
        len(menu_breadth),
        len(official_menu),
        len(official_capabilities),
        len(individual_ratings)
    ]
})

final_row_counts["rows_preserved"] = (
    final_row_counts["raw_rows"]
    == final_row_counts["cleaned_rows"]
)

final_row_counts

,dataset,raw_rows,cleaned_rows,rows_preserved
0,reviews,282,282,True
1,menu_prices,150,150,True
2,store_footprint,3,3,True
3,channels,16,16,True
4,menu_breadth,3,3,True
5,official_menu,36,36,True
6,official_capabilities,15,15,True
7,individual_ratings,13,13,True


In [87]:
print("FINAL CLEANING VALIDATION")
print("-" * 40)

print(
    "Reviews:",
    len(reviews)
)

print(
    "Usable review texts:",
    reviews["review_text_clean"].notna().sum()
)

print(
    "Standardized atomic themes:",
    len(standard_atomic_themes)
)

print(
    "Management dimensions:",
    len(dimension_columns)
)

print(
    "Menu observations:",
    len(menu_prices)
)

print(
    "Matched product families:",
    len(three_brand_families)
)

print(
    "Matched representative prices:",
    len(matched_family_prices)
)

print(
    "Unique branch/platform ratings:",
    len(branch_rating_records)
)

print(
    "Individual ratings:",
    len(individual_ratings)
)

print(
    "RQ2 inferential testing eligible:",
    RATING_HYPOTHESIS_TEST_ELIGIBLE
)

FINAL CLEANING VALIDATION
----------------------------------------
Reviews: 282
Usable review texts: 282
Standardized atomic themes: 25
Management dimensions: 8
Menu observations: 150
Matched product families: 5
Matched representative prices: 15
Unique branch/platform ratings: 55
Individual ratings: 13
RQ2 inferential testing eligible: False


## Missing Data Treatment

Missing values were retained when their absence carried analytical meaning or when no defensible basis for imputation existed.

- **Review dates:** 249 of 282 review dates were unavailable. These values were not imputed. Only the 33 observed dates were converted into a parsed date field.
- **Promotional prices:** Missing promotional prices indicate that no promotional price was observed for the product. These values were not replaced with zero or estimated values.
- **Discount values:** Discount amount and discount percentage remain missing when no promotional price was observed.
- **Comparable product family:** Products outside the five defensible cross-brand matched families remain unclassified rather than being forced into inappropriate comparisons.

No statistical imputation was required for the principal analytical variables.

## Individual Rating Limitation

Thirteen verified individual customer ratings were available for But First, Coffee across two branches.

Comparable individual-level ratings were not available for Starbucks and The Coffee Bean & Tea Leaf within the collected dataset. Consequently, the planned cross-brand inferential test for Research Question 2 cannot be performed without violating comparability requirements.

The available But First, Coffee individual ratings will therefore be used for descriptive analysis only.

Branch-level aggregate platform ratings are not substituted for individual ratings because they represent different units of analysis and are repeated across multiple written-review records.

In [88]:
reviews.to_csv(
    DATA_CLEANED / "reviews_cleaned.csv",
    index=False
)

menu_prices.to_csv(
    DATA_CLEANED / "menu_prices_cleaned.csv",
    index=False
)

store_footprint.to_csv(
    DATA_CLEANED / "store_footprint_cleaned.csv",
    index=False
)

channels.to_csv(
    DATA_CLEANED / "channels_partnerships_cleaned.csv",
    index=False
)

menu_breadth.to_csv(
    DATA_CLEANED / "menu_breadth_cleaned.csv",
    index=False
)

official_menu.to_csv(
    DATA_CLEANED / "official_menu_cleaned.csv",
    index=False
)

official_capabilities.to_csv(
    DATA_CLEANED / "official_capabilities_cleaned.csv",
    index=False
)

individual_ratings.to_csv(
    DATA_CLEANED / "individual_ratings_cleaned.csv",
    index=False
)

print("Core cleaned datasets exported successfully.")

Core cleaned datasets exported successfully.


In [89]:
matched_family_prices.to_csv(
    DATA_CLEANED / "matched_family_prices.csv",
    index=False
)

branch_rating_records.to_csv(
    DATA_CLEANED / "branch_rating_records.csv",
    index=False
)

theme_brand_prevalence.to_csv(
    DATA_CLEANED / "theme_brand_prevalence.csv",
    index=False
)

dimension_prevalence.to_csv(
    DATA_CLEANED / "dimension_prevalence.csv"
)

print("Analytical supporting tables exported successfully.")

Analytical supporting tables exported successfully.


In [90]:
exported_files = sorted(
    file.name
    for file in DATA_CLEANED.glob("*.csv")
)

print(
    f"Exported CSV files: "
    f"{len(exported_files)}"
)

for file in exported_files:
    print(f"- {file}")

Exported CSV files: 12
- branch_rating_records.csv
- channels_partnerships_cleaned.csv
- dimension_prevalence.csv
- individual_ratings_cleaned.csv
- matched_family_prices.csv
- menu_breadth_cleaned.csv
- menu_prices_cleaned.csv
- official_capabilities_cleaned.csv
- official_menu_cleaned.csv
- reviews_cleaned.csv
- store_footprint_cleaned.csv
- theme_brand_prevalence.csv


# Data Cleaning Conclusion

The raw competitive dataset was cleaned and transformed without altering the original source workbook.

Key cleaning outcomes include:

- 282 customer-review records retained and validated.
- 150 menu-pricing observations retained.
- Original menu categories preserved and standardized analytical categories created.
- Five defensible cross-brand comparable product families identified: Americano, Cafe Latte, Caramel Macchiato, Matcha Latte, and Mocha.
- A matched pricing dataset containing 15 representative brand-product-family observations created for subsequent statistical analysis.
- Promotional pricing separated from regular pricing. The observed But First, Coffee promotional items consistently reflected a 20% row-level discount.
- 282 usable customer-review texts retained.
- 73 original combined theme labels decomposed and standardized into 25 atomic customer-experience themes.
- Multi-label theme encoding retained reviews containing more than one customer-experience theme.
- Eight management-level customer-experience dimensions created for executive analysis.
- Rating-volume strings converted into analytical lower-bound fields while preserving their original values.
- 55 unique branch/platform aggregate-rating records identified.
- Thirteen individual But First, Coffee ratings retained for descriptive analysis.
- Cross-brand individual-rating hypothesis testing was classified as ineligible because comparable Starbucks and The Coffee Bean & Tea Leaf individual-rating observations were unavailable.
- Missing review dates, promotional prices, and unmatched product-family classifications were retained rather than artificially imputed.
- All source rows were preserved during the cleaning process.

The cleaned datasets are now ready for exploratory data analysis.